# Tool 3  Zoom-and-Reanalyze
Uses GradCAM++ heatmap from the trained OCTNet to crop the most activated region,
then re-runs the classifier on that crop at full resolution.

**Depends on:** `oct_best.pth` checkpoint and OCTNet architecture from your existing notebook.

**Pipeline:**
```
Image → GradCAM++ heatmap → threshold + bounding box → crop → resize 224×224 → OCTNet → new prediction
```

## 0. Paste OCTNet architecture here
We copy cells 0–4 from the main notebook (imports, config, OCTNet class definition).
Then load the checkpoint:

In [ ]:
import os, random, platform
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# CRITICAL: deterministic=True disables cuDNN autotuner 30% slower. Keep False.
# benchmark=True lets cuDNN pick fastest conv algo for your fixed input shape.
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IS_WIN = platform.system() == 'Windows'
print(f'Device: {DEVICE} | Platform: {platform.system()}')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {total_vram:.1f} GB')

In [ ]:
DATA_ROOT = Path('OCT2017')   
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR  = DATA_ROOT / 'test'

IMG_SIZE         = 224
BATCH_SIZE       = 8  # 128 @ 224×224 AMP fits in 6GB VRAM; drop to 64 if OOM
NUM_CLASSES      = 4
EPOCHS           = 60
LR               = 3e-4
WEIGHT_DECAY     = 1e-4
LABEL_SMOOTHING  = 0.05
WARMUP_EPOCHS    = 5
GRAD_CLIP        = 1.0
CLASS_NAMES      = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CKPT_PATH        = 'oct_best.pth'

# Windows: num_workers > 0 causes multiprocessing spawn overhead → use 0
# Linux:   4 workers with persistent_workers gives best throughput
NUM_WORKERS = 0 if IS_WIN else 4
PIN_MEMORY  = not IS_WIN and DEVICE.type == 'cuda'  # only useful with async workers
PERSISTENT  = NUM_WORKERS > 0
PREFETCH    = 2 if NUM_WORKERS > 0 else None

print(f'Batch: {BATCH_SIZE} | Workers: {NUM_WORKERS} | pin_memory: {PIN_MEMORY}')

In [ ]:
class DSConv(nn.Module):
    """Depthwise separable conv: dw → pw → BN → SiLU."""
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, k, padding=p, groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class AnisoBranch(nn.Module):
    """
    1×7 + 7×1 depthwise separable convolutions.
    Captures horizontal retinal layer continuity (1×7)
    and vertical cross-section disruptions (7×1).
    Outputs summed to keep channel count stable.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.h = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, (1,7), padding=(0,3), groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False))
        self.v = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, (7,1), padding=(3,0), groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False))
        self.bn  = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.h(x) + self.v(x)))


class CBAM(nn.Module):
    """Channel + Spatial attention (Woo et al. 2018)."""
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 4)
        self.ca_mlp = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(inplace=True), nn.Linear(mid, ch, bias=False))
        self.sa_conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)

    def forward(self, x):
        # Channel attention
        avg = x.mean([2,3]); mx = x.amax([2,3])
        ca  = torch.sigmoid(self.ca_mlp(avg) + self.ca_mlp(mx))
        x   = x * ca.unsqueeze(-1).unsqueeze(-1)
        # Spatial attention
        sa  = torch.sigmoid(self.sa_conv(torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], dim=1)))
        return x * sa


class MSBlock(nn.Module):
    """Multi-scale block: 3 branches → concat → CBAM → residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        bc = out_ch // 3
        ex = out_ch - 3 * bc
        self.b3 = DSConv(in_ch, bc + ex, k=3, p=1)
        self.b5 = DSConv(in_ch, bc,      k=5, p=2)
        self.ba = AnisoBranch(in_ch, bc)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.cbam = CBAM(out_ch)
        self.res  = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)) \
                    if in_ch != out_ch else nn.Identity()
        self.act  = nn.SiLU(inplace=True)

    def forward(self, x):
        out = self.cbam(self.bn(torch.cat([self.b3(x), self.b5(x), self.ba(x)], dim=1)))
        return self.act(out + self.res(x))


class OCTNet(nn.Module):
    def __init__(self, num_classes=4, drop=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True),  # 112
            nn.Conv2d(32, 64, 3, padding=1, bias=False),           nn.BatchNorm2d(64), nn.SiLU(inplace=True),
        )
        self.s1 = nn.Sequential(MSBlock(64,  96),  nn.MaxPool2d(2), nn.Dropout2d(0.10))  # 56
        self.s2 = nn.Sequential(MSBlock(96,  192), MSBlock(192,192), nn.MaxPool2d(2), nn.Dropout2d(0.15))  # 28
        self.s3 = nn.Sequential(MSBlock(192, 384), MSBlock(384,384), nn.MaxPool2d(2), nn.Dropout2d(0.20))  # 14
        self.s4 = nn.Sequential(MSBlock(384, 512), nn.MaxPool2d(2))  # 7
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.SiLU(inplace=True),
            nn.Dropout(drop), nn.Linear(256, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear) and m.weight is not None:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.s4(self.s3(self.s2(self.s1(self.stem(x))))))

    @property
    def cam_layer(self):
        """Last depthwise conv — target for GradCAM++."""
        return self.s4[0].b3.net[0]


model = OCTNet(NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} ({n_params/1e6:.2f}M)')
assert n_params < 5_000_000, f'Too many params: {n_params:,}'

# torch.compile disabled — requires CUDA dev headers (triton build fails without them)
# Re-enable later with: sudo apt install nvidia-cuda-toolkit
import torch._dynamo
torch._dynamo.config.suppress_errors = True
print('torch.compile: disabled')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    print(f'VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
class GradCAMpp:
    """GradCAM++ no external library required."""
    def __init__(self, model, layer):
        self.model = model; self.grads = None; self.acts = None
        layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'grads',go[0].detach()))

    def __call__(self, img_t, cls=None):
        self.model.eval()
        x = img_t.unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = self.model(x)
        cls = cls if cls is not None else logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, cls].backward()
        g, a = self.grads[0], self.acts[0]            # (C,H,W)
        g2, g3 = g**2, g**3
        alpha = g2 / (2*g2 + (a*g3).sum([1,2], keepdim=True) + 1e-7)
        w = (alpha * F.relu(g)).sum([1,2])             # (C,)
        cam = F.relu((w[:,None,None] * a).sum(0))
        cam = (cam - cam.min()) / (cam.max() + 1e-7)
        conf = torch.softmax(logits, 1)[0].detach().cpu().numpy()
        return cam.cpu().numpy(), cls, conf


# Load test dataset for later use
val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
test_ds = datasets.ImageFolder(TEST_DIR, transform=val_tf)

eval_model = OCTNet(NUM_CLASSES).to(DEVICE)
eval_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
eval_model.eval()
cam_fn = GradCAMpp(eval_model, eval_model.cam_layer)
print('Model and GradCAM++ ready.')

## 1. Zoom-and-Reanalyze Tool

In [ ]:
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms


def get_cam_bbox(cam: np.ndarray, threshold: float = 0.6) -> tuple[int, int, int, int]:
    """
    Threshold the GradCAM++ heatmap and return the tight bounding box
    (x0, y0, x1, y1) of the activated region.
    threshold: fraction of max activation to use as cutoff (0.4 = top 60% of signal)
    Returns pixel coords in the CAM's own resolution.
    """
    binary = (cam >= threshold).astype(np.uint8)
    rows = np.any(binary, axis=1)
    cols = np.any(binary, axis=0)

    # Guard: if threshold too high, fall back to centre crop
    if not rows.any() or not cols.any():
        h, w = cam.shape
        return w // 4, h // 4, 3 * w // 4, 3 * h // 4

    y0, y1 = np.where(rows)[0][[0, -1]]
    x0, x1 = np.where(cols)[0][[0, -1]]
    return int(x0), int(y0), int(x1), int(y1)


def zoom_and_reanalyze(
    pil_image: Image.Image,
    model,
    cam_fn,
    class_names: list[str],
    img_size: int = 224,
    cam_threshold: float = 0.4,
    padding_frac: float = 0.10,
) -> dict:
    """
    Full zoom-and-reanalyze pipeline.

    Args:
        pil_image:      PIL image (any size/mode — will be converted internally).
        model:          Loaded OCTNet in eval mode.
        cam_fn:         GradCAMpp instance bound to the model.
        class_names:    e.g. ['CNV', 'DME', 'DRUSEN', 'NORMAL']
        img_size:       Model input size (224).
        cam_threshold:  Fraction of peak activation used to define the ROI.
        padding_frac:   Extra context added around the bbox (10% of image dims).

    Returns dict with keys:
        original_pred, original_conf, original_probs  — first pass on full image
        cam                                           — heatmap (H×W numpy)
        bbox                                          — (x0,y0,x1,y1) in image pixels
        crop_pred, crop_conf, crop_probs              — second pass on crop
        agreement                                     — True if both passes agree
    """
    # Inference transform (no augmentation) 
    infer_tf = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    # Pass 1: full image 
    img_t = infer_tf(pil_image)                 # (3, H, W)
    cam, orig_cls, orig_conf = cam_fn(img_t)    # cam is (cam_h, cam_w) in [0,1]

    #  Locate ROI 
    # Scale CAM coords → full img_size pixel coords
    cam_h, cam_w = cam.shape
    scale_x = img_size / cam_w
    scale_y = img_size / cam_h

    cx0, cy0, cx1, cy1 = get_cam_bbox(cam, cam_threshold)
    x0 = int(cx0 * scale_x)
    y0 = int(cy0 * scale_y)
    x1 = int(cx1 * scale_x)
    y1 = int(cy1 * scale_y)

    # Add padding so we don't lose context at the boundary
    pad_x = int(img_size * padding_frac)
    pad_y = int(img_size * padding_frac)
    x0 = max(0, x0 - pad_x)
    y0 = max(0, y0 - pad_y)
    x1 = min(img_size, x1 + pad_x)
    y1 = min(img_size, y1 + pad_y)

    # Crop from the resized tensor, then re-run 
    # img_t is already (3, img_size, img_size); slice and upsample back to img_size
    crop_t = img_t[:, y0:y1, x0:x1].unsqueeze(0)          # (1, 3, crop_h, crop_w)
    crop_t = F.interpolate(crop_t, size=(img_size, img_size), mode='bilinear', align_corners=False)
    crop_t = crop_t.squeeze(0)                              # (3, img_size, img_size)

    # Pass 2: cropped region 
    _, crop_cls, crop_conf = cam_fn(crop_t)

    return {
        # Pass 1
        'original_pred':  class_names[orig_cls],
        'original_conf':  float(orig_conf[orig_cls]),
        'original_probs': {c: float(p) for c, p in zip(class_names, orig_conf)},
        # Localization
        'cam':  cam,
        'bbox': (x0, y0, x1, y1),
        # Pass 2
        'crop_pred':  class_names[crop_cls],
        'crop_conf':  float(crop_conf[crop_cls]),
        'crop_probs': {c: float(p) for c, p in zip(class_names, crop_conf)},
        # Meta
        'agreement': orig_cls == crop_cls,
    }


print('zoom_and_reanalyze() defined.')

## 2. Visualize a single example

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def visualize_zoom(pil_image: Image.Image, result: dict, class_names: list[str], img_size: int = 224):
    """4-panel figure: original | CAM overlay | annotated bbox | crop reanalysis."""
    # Prepare numpy versions
    img_np = np.array(pil_image.convert('L').resize((img_size, img_size))) / 255.
    cam_up = np.array(
        Image.fromarray((result['cam'] * 255).astype(np.uint8)).resize((img_size, img_size), Image.BILINEAR)
    ) / 255.
    overlay = (0.55 * np.stack([img_np]*3, 2) + 0.45 * plt.cm.jet(cam_up)[:, :, :3]).clip(0, 1)

    x0, y0, x1, y1 = result['bbox']
    crop_np = img_np[y0:y1, x0:x1]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f"Zoom-and-Reanalyze  |  "
        f"Pass 1: {result['original_pred']} ({result['original_conf']*100:.1f}%)  →  "
        f"Pass 2: {result['crop_pred']} ({result['crop_conf']*100:.1f}%)  "
        f"{'✓ agree' if result['agreement'] else '✗ disagree'}",
        fontsize=12
    )

    axes[0].imshow(img_np, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(overlay);             axes[1].set_title('GradCAM++ overlay'); axes[1].axis('off')

    axes[2].imshow(img_np, cmap='gray')
    rect = patches.Rectangle((x0, y0), x1-x0, y1-y0, linewidth=2, edgecolor='red', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].set_title('ROI bounding box'); axes[2].axis('off')

    axes[3].imshow(crop_np, cmap='gray')
    axes[3].set_title(f"Crop → {result['crop_pred']} ({result['crop_conf']*100:.1f}%)"); axes[3].axis('off')

    plt.tight_layout()
    plt.savefig('tool3_zoom_example.png', dpi=150, bbox_inches='tight')
    plt.show()


# Run on one test image 
# Load any PIL image — here we grab one from the test set
sample_path, sample_label = test_ds.samples[5]
pil_img = Image.open(sample_path)

result = zoom_and_reanalyze(pil_img, eval_model, cam_fn, CLASS_NAMES)
print(f"Pass 1 : {result['original_pred']} ({result['original_conf']*100:.1f}%)")
print(f"Bbox   : {result['bbox']}")
print(f"Pass 2 : {result['crop_pred']} ({result['crop_conf']*100:.1f}%)")
print(f"Agree  : {result['agreement']}")

visualize_zoom(pil_img, result, CLASS_NAMES)

## 3. Batch evaluation: does zooming change any predictions?
Run on the whole test set and report disagreement rate and accuracy of each pass.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

true_labels, orig_preds, crop_preds = [], [], []

for path, label in tqdm(test_ds.samples, desc='Zoom eval'):
    pil_img = Image.open(path)
    r = zoom_and_reanalyze(pil_img, eval_model, cam_fn, CLASS_NAMES)
    true_labels.append(label)
    orig_preds.append(CLASS_NAMES.index(r['original_pred']))
    crop_preds.append(CLASS_NAMES.index(r['crop_pred']))

acc_orig = accuracy_score(true_labels, orig_preds)
acc_crop = accuracy_score(true_labels, crop_preds)
disagree = sum(o != c for o, c in zip(orig_preds, crop_preds))

print(f'\nPass 1 (full image) accuracy : {acc_orig*100:.2f}%')
print(f'Pass 2 (crop)       accuracy : {acc_crop*100:.2f}%')
print(f'Disagreements                : {disagree}/{len(true_labels)} ({disagree/len(true_labels)*100:.1f}%)')
print()
print('Pass 2 (crop) classification report')
print(classification_report(true_labels, crop_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
OCTID_DIR = Path('octid')
octid_ds = datasets.ImageFolder(OCTID_DIR, transform=val_tf)
OCTID_CLASSES = sorted(octid_ds.classes)
print(f'OCTID classes: {OCTID_CLASSES} | n={len(octid_ds)}')

# Run zoom-and-reanalyze on OCTID
print('\n' + '='*70)
print('OCTID Dataset Zoom-and-Reanalyze Evaluation')
print('='*70)
octid_labels, octid_orig_preds, octid_crop_preds = [], [], []

for path, label in tqdm(octid_ds.samples[:100], desc='OCTID zoom eval'):  # Limit to 100 for speed
    pil_img = Image.open(path)
    r = zoom_and_reanalyze(pil_img, eval_model, cam_fn, CLASS_NAMES)
    octid_labels.append(label)
    octid_orig_preds.append(CLASS_NAMES.index(r['original_pred']))
    octid_crop_preds.append(CLASS_NAMES.index(r['crop_pred']))

octid_acc_orig = accuracy_score(octid_labels, octid_orig_preds)
octid_acc_crop = accuracy_score(octid_labels, octid_crop_preds)
octid_disagree = sum(o != c for o, c in zip(octid_orig_preds, octid_crop_preds))

print(f'\nOCTID Results (n=100 samples):')
print(f'Pass 1 (full) accuracy: {octid_acc_orig*100:.2f}%')
print(f'Pass 2 (crop) accuracy: {octid_acc_crop*100:.2f}%')
print(f'Disagreements:          {octid_disagree}/100 ({octid_disagree}%)')

# Load OCT-C8
C8_TEST_DIR = Path('octc8/test')
c8_ds = datasets.ImageFolder(C8_TEST_DIR, transform=val_tf)
C8_CLASSES = sorted(c8_ds.classes)
print(f'\nOCT-C8 classes: {C8_CLASSES} | n={len(c8_ds)}')

# Run zoom-and-reanalyze on OCT-C8
print('\n' + '='*70)
print('OCT-C8 Dataset Zoom-and-Reanalyze Evaluation')
print('='*70)
c8_labels, c8_orig_preds, c8_crop_preds = [], [], []

for path, label in tqdm(c8_ds.samples[:100], desc='C8 zoom eval'):  # Limit to 100 for speed
    pil_img = Image.open(path)
    r = zoom_and_reanalyze(pil_img, eval_model, cam_fn, CLASS_NAMES)
    c8_labels.append(label)
    c8_orig_preds.append(CLASS_NAMES.index(r['original_pred']))
    c8_crop_preds.append(CLASS_NAMES.index(r['crop_pred']))

c8_acc_orig = accuracy_score(c8_labels, c8_orig_preds)
c8_acc_crop = accuracy_score(c8_labels, c8_crop_preds)
c8_disagree = sum(o != c for o, c in zip(c8_orig_preds, c8_crop_preds))

print(f'\nOCT-C8 Results (n=100 samples):')
print(f'Pass 1 (full) accuracy: {c8_acc_orig*100:.2f}%')
print(f'Pass 2 (crop) accuracy: {c8_acc_crop*100:.2f}%')
print(f'Disagreements:          {c8_disagree}/100 ({c8_disagree}%)')

print('\n' + '='*70)
print('Summary: Cross-Dataset Zoom-and-Reanalyze')
print('='*70)
print(f'OCTID:  Pass 1 {octid_acc_orig*100:.2f}% → Pass 2 {octid_acc_crop*100:.2f}% (Δ {(octid_acc_crop-octid_acc_orig)*100:+.2f}pp)')
print(f'OCT-C8: Pass 1 {c8_acc_orig*100:.2f}% → Pass 2 {c8_acc_crop*100:.2f}% (Δ {(c8_acc_crop-c8_acc_orig)*100:+.2f}pp)')